In [ ]:
# Install necessary libraries
# !pip install transformers datasets PIL pytorch

In [ ]:
import torch
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor
from PIL import Image
import time
import json
from datasets import load_dataset

In [ ]:
# Login to Hugging Face (if required)
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
# Load the model and preprocessor
model_id = "oportunitas/rt-detr-r18-meals"
model = RTDetrForObjectDetection.from_pretrained(model_id)
processor = RTDetrImageProcessor.from_pretrained(model_id)

In [ ]:
# Load the test dataset
dataset = load_dataset("coco", data_dir="data/coco/test")
dataset = dataset['train'] # Use train split as it contains the annotations

In [ ]:
# Prepare the data for inference
def preprocess_image(image):
    inputs = processor(images=image, return_tensors="pt")
    return inputs


In [ ]:
# Perform inference and measure time
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

inference_times = []

with torch.no_grad():
    for i in range(len(dataset)):
        image = dataset[i]["image"]
        inputs = preprocess_image(image).to(device)

        start_time = time.time()
        outputs = model(**inputs)
        end_time = time.time()

inference_time = end_time - start_time
        inference_times.append(inference_time)

In [ ]:
# Calculate average inference time and FPS
average_inference_time = sum(inference_times) / len(inference_times)
fps = 1 / average_inference_time

# Print the results
print(f"Average Inference Time: {average_inference_time:.4f} seconds")
print(f"FPS: {fps:.2f}")